#### EDA (Summary Stats, Univariate, Bivariate, Outliers) ###

In [34]:
print("products:", type(products))
print("orders:", type(orders))
print("order_items:", type(order_items))
print("customers:", type(customers))
print("sellers:", type(sellers))
print("product_category:", type(product_category))


products: <class 'pandas.DataFrame'>
orders: <class 'pandas.DataFrame'>


NameError: name 'order_items' is not defined

In [ ]:
# Clean duplicated columns that often appear after repeated merges
products = products.loc[:, ~products.columns.duplicated()].copy()

# Keep only unique IDs for merge keys
products_for_merge = (
    products[["product_id", "product_category_name_english", "product_weight_g"]]
    .drop_duplicates(subset=["product_id"])
    .copy()
)

customers_for_merge = (
    customers[["customer_id", "customer_city", "customer_state"]]
    .drop_duplicates(subset=["customer_id"])
    .copy()
)

sellers_for_merge = (
    sellers[["seller_id", "seller_city", "seller_state"]]
    .drop_duplicates(subset=["seller_id"])
    .copy()
)

# Convert date columns safely
for col in date_columns:
    if col in orders.columns:
        orders[col] = pd.to_datetime(orders[col], errors="coerce")

# Rebuild master_df safely
master_df = order_items.merge(orders, on="order_id", how="inner")
master_df = master_df.merge(products_for_merge, on="product_id", how="left")
master_df = master_df.merge(customers_for_merge, on="customer_id", how="left")
master_df = master_df.merge(sellers_for_merge, on="seller_id", how="left")

# Compute delivery time in days
master_df["delivery_time_days"] = (
    pd.to_datetime(master_df["order_delivered_customer_date"], errors="coerce")
    - pd.to_datetime(master_df["order_purchase_timestamp"], errors="coerce")
).dt.days

print("master_df shape:", master_df.shape)
print(master_df.head())

In [33]:
import pandas as pd
import numpy as np

print("=== STEP 1: CREATE MASTER_DF STARTED ===")

# 1. Merge product category translation into products
products = products.merge(product_category, on="product_category_name", how="left")

# 2. Start master_df from order_items + orders
master_df = order_items.merge(orders, on="order_id", how="inner")

# 3. Add product info (category + weight)
master_df = master_df.merge(
    products[["product_id", "product_category_name_english", "product_weight_g"]],
    on="product_id",
    how="left"
)

# 4. Add customer info (city, state)
master_df = master_df.merge(
    customers[["customer_id", "customer_city", "customer_state"]],
    on="customer_id",
    how="left"
)

# 5. Add seller info (city, state)
master_df = master_df.merge(
    sellers[["seller_id", "seller_city", "seller_state"]],
    on="seller_id",
    how="left"
)

print("master_df created successfully.")
print("Shape:", master_df.shape)
print(master_df.head())
print("=== STEP 1 COMPLETED ===")


=== STEP 1: CREATE MASTER_DF STARTED ===


MergeError: Passing 'suffixes' which cause duplicate columns {'product_category_name_english_x', 'product_category_name_english_y'} is not allowed.